In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS automobile_repair.silver;

CREATE OR REPLACE TABLE automobile_repair.silver.estimate AS
WITH deduplicated_estimates AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY estimate_id ORDER BY created_at DESC) as rn
    FROM automobile_repair.bronze.estimate
),
base_estimates AS (
    SELECT
        estimate_id,
        order_id,
        version_no,
        estimate_amount,
        CAST(created_at AS DATE) AS estimate_date,
        estimate_type,
        estimator_id,
        estimator_name
    FROM deduplicated_estimates
    WHERE rn = 1
),
version_metrics AS (
    SELECT 
        order_id,
        MAX(version_no) as max_version_no
    FROM base_estimates
    GROUP BY order_id
),
initial_estimates AS (
    SELECT 
        order_id,
        estimate_amount as initial_estimate
    FROM base_estimates
    WHERE version_no = 1
),
final_estimates AS (
    SELECT 
        b.order_id,
        b.estimate_amount as final_estimate
    FROM base_estimates b
    INNER JOIN version_metrics v 
        ON b.order_id = v.order_id 
        AND b.version_no = v.max_version_no
)
SELECT 
    -- Identifiers (2)
    b.estimate_id,
    b.order_id,
    
    -- Estimator info (3)
    b.estimator_id,
    b.estimator_name,
    b.estimate_type,
    
    -- Classification (1)
    CASE 
        WHEN b.version_no = 1 THEN 'Initial'
        WHEN b.version_no = v.max_version_no THEN 'Final'
        ELSE 'Intermediate'
    END AS version_type,
    
    -- Amounts (2)
    i.initial_estimate,
    f.final_estimate,
    
    -- Accuracy metrics (2) - only populated on Final estimates
    CASE 
        WHEN b.version_no = v.max_version_no 
             AND i.initial_estimate IS NOT NULL 
             AND f.final_estimate IS NOT NULL
             AND i.initial_estimate > 0
        THEN ROUND(100 - ABS((f.final_estimate - i.initial_estimate) * 100.0 / i.initial_estimate), 2)
        ELSE NULL
    END AS accuracy_score,
    
    CASE 
        WHEN b.version_no = v.max_version_no 
             AND i.initial_estimate IS NOT NULL 
             AND f.final_estimate IS NOT NULL
             AND i.initial_estimate > 0
        THEN ROUND((f.final_estimate - i.initial_estimate) * 100.0 / i.initial_estimate, 2)
        ELSE NULL
    END AS variance_pct,
    
    -- Date (1)
    b.estimate_date
    
FROM base_estimates b
LEFT JOIN version_metrics v ON b.order_id = v.order_id
LEFT JOIN initial_estimates i ON b.order_id = i.order_id
LEFT JOIN final_estimates f ON b.order_id = f.order_id
WHERE b.estimate_amount IS NOT NULL  -- Exclude NULL amounts
ORDER BY b.estimate_date DESC, b.order_id, b.version_no;